執行順序：掃描（可選）→ 連線 → 能力檢查 → 讀取範圍 → 取得控制權 → 選擇一種控制模式 → 斷線。

固定阻力與 App 0–30 ERG 只能選一種；最後執行的模式會覆蓋前一種。

In [1]:
import asyncio
import struct
from dataclasses import dataclass
from bleak import BleakClient, BleakScanner

FTMS_SERVICE = '00001826-0000-1000-8000-00805f9b34fb'
CONTROL_POINT = '00002ad9-0000-1000-8000-00805f9b34fb'
SUPPORTED_RESISTANCE_RANGE = '00002ad6-0000-1000-8000-00805f9b34fb'
SUPPORTED_POWER_RANGE = '00002ad8-0000-1000-8000-00805f9b34fb'
RESPONSE_CODE = 0x80
SUCCESS = 0x01

@dataclass(frozen=True)
class NumericRange:
    minimum: float
    maximum: float
    increment: float

    def align(self, value: float) -> float:
        steps = round((value - self.minimum) / self.increment)
        return self.minimum + steps * self.increment

class T500Ftms:
    def __init__(self, address: str, timeout: float = 15) -> None:
        self.address = address
        self.timeout = timeout
        self.client = None
        self.control_event = asyncio.Event()
        self.control_response = None
        self.notifications_active = False

    async def connect(self) -> None:
        device = await BleakScanner.find_device_by_address(self.address, timeout=self.timeout)
        if device is None:
            raise RuntimeError(f'Trainer {self.address} was not found. Wake it, then try again.')
        self.client = BleakClient(
            device,
            timeout=self.timeout,
            services=[FTMS_SERVICE],
            winrt={'use_cached_services': False},
        )
        await self.client.connect()
        print(f'Connected: {self.client.is_connected}')

    async def disconnect(self) -> None:
        if self.client is not None and self.client.is_connected:
            await self.client.disconnect()
        self.notifications_active = False
        print('Disconnected')

    def _require(self, uuid: str, name: str) -> None:
        if self.client is None or self.client.services.get_characteristic(uuid) is None:
            raise RuntimeError(
                f'{name} is missing. Wake the trainer, close other apps, then reconnect.'
            )

    async def capabilities(self, timeout: float = 10) -> dict[str, bool]:
        checks = {
            'Control Point': CONTROL_POINT,
            'Resistance Range': SUPPORTED_RESISTANCE_RANGE,
            'Power Range': SUPPORTED_POWER_RANGE,
        }
        deadline = asyncio.get_running_loop().time() + timeout
        while True:
            result = {
                name: self.client.services.get_characteristic(uuid) is not None
                for name, uuid in checks.items()
            }
            if all(result.values()) or asyncio.get_running_loop().time() >= deadline:
                break
            await asyncio.sleep(0.25)
        for name, available in result.items():
            print(f'{name}: {"available" if available else "missing"}')
        return result

    async def read_resistance_range(self) -> NumericRange:
        self._require(SUPPORTED_RESISTANCE_RANGE, 'Resistance Range')
        raw = await self.client.read_gatt_char(SUPPORTED_RESISTANCE_RANGE)
        minimum, maximum, increment = struct.unpack('<hhh', raw)
        result = NumericRange(minimum / 10, maximum / 10, increment / 10)
        print(f'Resistance range: {result.minimum:g} to {result.maximum:g}, step {result.increment:g}')
        return result

    async def read_power_range(self) -> NumericRange:
        self._require(SUPPORTED_POWER_RANGE, 'Power Range')
        raw = await self.client.read_gatt_char(SUPPORTED_POWER_RANGE)
        minimum, maximum, increment = struct.unpack('<hhh', raw)
        result = NumericRange(minimum, maximum, increment)
        print(f'Power range: {result.minimum:g} to {result.maximum:g} W, step {result.increment:g} W')
        return result

    def _on_control_point(self, _sender: int, data: bytearray) -> None:
        if len(data) >= 3 and data[0] == RESPONSE_CODE:
            self.control_response = (data[1], data[2])
            self.control_event.set()

    async def _send(self, payload: bytes) -> None:
        self.control_response = None
        self.control_event.clear()
        print(f'TX Control Point: {payload.hex(" ")}')
        await self.client.write_gatt_char(CONTROL_POINT, payload, response=True)
        await asyncio.wait_for(self.control_event.wait(), timeout=5)
        if self.control_response is None:
            raise RuntimeError('No Control Point response received.')
        opcode, result = self.control_response
        print(f'RX Control Point: opcode=0x{opcode:02x}, result=0x{result:02x}')
        if opcode != payload[0] or result != SUCCESS:
            raise RuntimeError(f'Command rejected: opcode=0x{payload[0]:02x}, result=0x{result:02x}')

    async def acquire_control(self) -> None:
        self._require(CONTROL_POINT, 'Control Point')
        if not self.notifications_active:
            await self.client.start_notify(CONTROL_POINT, self._on_control_point)
            self.notifications_active = True
        await self._send(b'\x00')
        await self._send(b'\x07')
        print('Trainer control acquired')

    async def set_resistance(self, level: float) -> float:
        supported = await self.read_resistance_range()
        if not supported.minimum <= level <= supported.maximum:
            raise ValueError(f'Resistance must be {supported.minimum:g} to {supported.maximum:g}.')
        actual = supported.align(level)
        await self._send(struct.pack('<Bh', 0x04, round(actual * 10)))
        print(f'Native resistance set to: {actual:g}')
        return actual

    async def set_app_level(self, level: int, level_30_watts: float = 300) -> float:
        if not 0 <= level <= 30:
            raise ValueError('App resistance level must be 0 to 30.')
        supported = await self.read_power_range()
        if not supported.minimum <= level_30_watts <= supported.maximum:
            raise ValueError(f'Level 30 must be {supported.minimum:g} to {supported.maximum:g} W.')
        target = supported.minimum + (level_30_watts - supported.minimum) * level / 30
        actual = supported.align(target)
        await self._send(struct.pack('<Bh', 0x05, round(actual)))
        print(f'App level: {level}/30; ERG target: {actual:g} W')
        return actual

async def scan(timeout: float = 10) -> list[tuple[str, str]]:
    devices = await BleakScanner.discover(timeout=timeout)
    return [(device.address, device.name or '(unknown)') for device in devices]


##### Scan (optional)

In [27]:
for address, name in await scan():
    print(f'{address}  {name}')

03:F6:B2:EF:DA:79  (unknown)
68:43:C3:A8:D0:C8  LE_WH-1000XM5
52:D3:1F:45:7E:5C  (unknown)
63:F6:46:4D:1B:0F  (unknown)
79:FB:0D:6D:AD:EB  (unknown)
69:37:04:96:7D:DD  (unknown)
1D:5B:48:14:10:64  (unknown)
6E:22:DC:F3:68:A2  (unknown)
1A:CB:8D:D6:EC:C6  (unknown)
35:D8:84:A0:D9:74  (unknown)
7B:19:B5:21:5E:D0  (unknown)
26:6A:0E:8E:85:22  (unknown)
E2:70:EA:45:9D:22  (unknown)
47:D1:87:F0:B3:B7  (unknown)
24:B7:6B:AA:73:90  (unknown)
D8:05:0A:4B:30:68  P715 64059 R
EC:40:14:CD:6F:8F  57_32736
29:87:2C:BE:EA:D7  (unknown)
70:48:BC:BD:C3:5A  (unknown)
3F:D1:14:F1:D3:83  (unknown)
DF:31:F0:88:7C:DE  (unknown)
4A:66:8C:16:49:E0  (unknown)
44:DD:A7:94:37:70  (unknown)
5A:94:FB:63:15:14  (unknown)
6A:54:EB:1C:11:7B  (unknown)
57:8B:5E:4E:ED:D7  (unknown)
5C:7E:90:03:4C:18  (unknown)
F5:98:42:C8:0A:DC  Mi Smart Band 5
FD:25:17:79:3B:6C  P715 54922


##### Connect

In [2]:
ADDRESS = 'EC:40:14:CD:6F:8F'
trainer = T500Ftms(ADDRESS)
await trainer.connect()

Connected: True


##### Check FTMS capabilities
所有項目都必須顯示 available。若顯示 missing，代表目前只有藍牙連線，尚未有可控制的 FTMS 服務。

In [3]:
capabilities = await trainer.capabilities()
assert all(capabilities.values()), 'FTMS capabilities are incomplete.'


Control Point: available
Resistance Range: available
Power Range: available


##### Read supported ranges

讀取 T500 實際支援的原生阻力與 ERG 功率範圍。

In [4]:
resistance_range = await trainer.read_resistance_range()
power_range = await trainer.read_power_range()


Resistance range: 0 to 10, step 0.1
Power range: 20 to 2200 W, step 1 W


##### Acquire control
這格會送出 FTMS Request Control 與 Start or Resume。每次寫入都會列出 TX/RX 控制點封包；result=0x01 代表命令被接受。

In [5]:
await trainer.acquire_control()

TX Control Point: 00
RX Control Point: opcode=0x00, result=0x01
TX Control Point: 07
RX Control Point: opcode=0x07, result=0x01
Trainer control acquired


##### Native resistance

設定 T500 原生阻力等級。先用 0 和 10 測試兩端差異。執行這格會覆蓋 ERG。

In [16]:
TARGET_RESISTANCE = 0.1
current_resistance = await trainer.set_resistance(TARGET_RESISTANCE)


Resistance range: 0 to 10, step 0.1
TX Control Point: 04 01 00
RX Control Point: opcode=0x04, result=0x01
Native resistance set to: 0.1


In [12]:
APP_RESISTANCE_LEVEL = 30  # 0 to 30
LEVEL_30_WATTS = 300      # Change only if 300 W is unsuitable for you
current_power = await trainer.set_app_level(APP_RESISTANCE_LEVEL,level_30_watts=LEVEL_30_WATTS,)

Power range: 20 to 2200 W, step 1 W
TX Control Point: 05 2c 01
RX Control Point: opcode=0x05, result=0x01
App level: 30/30; ERG target: 300 W


##### Disconnect

In [17]:
await trainer.disconnect()

Disconnected
